# Parallel Colebrook-White Friction Factor Solver with ParFun

The **Colebrook-White equation** is the industry-standard formula for computing the **Darcy-Weisbach friction factor** in turbulent pipe flow:

$$
\frac{1}{\sqrt{f}} = -2 \log_{10}\!\left(\frac{\varepsilon/D}{3.7} + \frac{2.51}{\text{Re}\sqrt{f}}\right)
$$

Because $f$ appears on both sides, there is no closed-form solution -- it must be solved numerically.
We solve it for **500,000 pipe segments** (a large municipal water network) using Brent's method with **20 initial guesses** each, giving **10,000,000 total solves**.

ParFun parallelises this with minimal code changes.

## Program Structure

```mermaid
flowchart TB
    subgraph seq["Sequential"]
        direction TB
        S1["Pipe 1 solve"] --> S2["Pipe 2 solve"]
        S2 --> S3["Pipe 3 solve"]
        S3 --> S4["..."] --> S5["Pipe 500,000 solve"]
    end

    subgraph par["Parallel 4 workers"]
        direction TB
        subgraph W1["Worker 1"]
            direction TB
            A1["Pipes 1 - 125K"]
        end
        subgraph W2["Worker 2"]
            direction TB
            A2["Pipes 125K - 250K"]
        end
        subgraph W3["Worker 3"]
            direction TB
            A3["Pipes 250K - 375K"]
        end
        subgraph W4["Worker 4"]
            direction TB
            A4["Pipes 375K - 500K"]
        end
        W1 & W2 & W3 & W4 --> C1(["concat results"])
    end
```

# Imports and Environment

In [1]:
import time
from typing import List, Optional

import numpy as np
from scipy.optimize import brentq

import parfun as pf

np.random.seed(42)


# Core physics: the Colebrook-White residual and single-pipe solver.

In [2]:
def colebrook_residual(f: float, Re: float, rel_roughness: float) -> float:
    """Returns g(f) = 0 at the true friction factor."""
    if f <= 0:
        return np.inf
    return 1.0 / np.sqrt(f) + 2.0 * np.log10(rel_roughness / 3.7 + 2.51 / (Re * np.sqrt(f)))


def solve_single_pipe(Re: float, rel_roughness: float, n_guesses: int = 20) -> Optional[float]:
    """Solve Colebrook-White for one pipe segment.

    Sweeps over n_guesses bracketed intervals and accepts the first convergent root.
    Returns the friction factor f, or NaN if no solution found.
    """
    f_candidates = np.logspace(np.log10(0.008), np.log10(0.1), n_guesses + 1)

    for i in range(len(f_candidates) - 1):
        fa = f_candidates[i]
        fb = f_candidates[i + 1]
        ga = colebrook_residual(fa, Re, rel_roughness)
        gb = colebrook_residual(fb, Re, rel_roughness)

        if np.isfinite(ga) and np.isfinite(gb) and ga * gb < 0:
            try:
                root = brentq(colebrook_residual, fa, fb,
                              args=(Re, rel_roughness), xtol=1e-10, maxiter=200)
                return root
            except Exception:
                continue
    return np.nan

# Function to Parallelize

In [3]:
def solve_network(
    re_list: List[float],
    roughness_list: List[float],
    n_guesses: int = 20,
) -> List[float]:
    """Solve Colebrook-White for every pipe segment."""
    results = []
    for Re, eps in zip(re_list, roughness_list):
        f = solve_single_pipe(Re, eps, n_guesses)
        results.append(f if f is not None else np.nan)
    return results

# Parallelizing with ParFun

The only change needed is wrapping `solve_network` with a `@pf.parfun` decorator.
The parallel version calls the **same function** — no logic is duplicated:

```diff
+ @pf.parfun(
+     split=pf.per_argument(
+         re_list=pf.py_list.by_chunk,
+         roughness_list=pf.py_list.by_chunk,
+     ),
+     combine_with=pf.py_list.concat,
+     fixed_partition_size=125_000,
+ )
  def solve_network_w_parfun(re_list, roughness_list, n_guesses=20):
+     return solve_network(re_list, roughness_list, n_guesses)
  ...
+ with pf.set_parallel_backend_context("scaler_local", n_workers=4):
+     result = solve_network_w_parfun(re_values, roughness_values, N_GUESSES)
```

In [4]:
@pf.parfun(
    split=pf.per_argument(
        re_list=pf.py_list.by_chunk,
        roughness_list=pf.py_list.by_chunk,
    ),
    combine_with=pf.py_list.concat,
    fixed_partition_size=125_000,
)
def solve_network_w_parfun(
    re_list: List[float],
    roughness_list: List[float],
    n_guesses: int = 20,
) -> List[float]:
    return solve_network(re_list, roughness_list, n_guesses)

In [5]:
N_PIPES = 500_000
N_GUESSES = 20

re_values = np.random.uniform(4_000, 2_000_000, N_PIPES).tolist()
roughness_values = np.random.uniform(0.0001, 0.05, N_PIPES).tolist()

print(f"Pipeline network: {N_PIPES:,} segments x {N_GUESSES} guesses = {N_PIPES * N_GUESSES:,} solves")

start = time.time()
serial_results = solve_network(re_values, roughness_values, N_GUESSES)
seq_time = time.time() - start

start = time.time()
with pf.set_parallel_backend_context("scaler_local", n_workers=4):
    parallel_results = solve_network_w_parfun(re_values, roughness_values, N_GUESSES)
par_time = time.time() - start

print(f"Sequential: {seq_time:.2f}s")
print(f"Parallel:   {par_time:.2f}s")
print(f"Speedup:    {seq_time / par_time:.2f}x")

serial_arr = np.array(serial_results)
parallel_arr = np.array(parallel_results)
max_diff = np.nanmax(np.abs(serial_arr - parallel_arr))
print(f"\nMax abs diff: {max_diff:.2e}")
n_solved = np.sum(~np.isnan(serial_arr))
print(f"Pipes solved: {n_solved:,} / {N_PIPES:,}")

Pipeline network: 500,000 segments x 20 guesses = 10,000,000 solves
[INFO]2026-03-28 02:44:25+0800: logging to ('/dev/stdout',)
[INFO]2026-03-28 02:44:25+0800: ObjectStorageServer: start and listen to tcp://127.0.0.1:51993
[INFO]2026-03-28 02:44:25+0800: ObjectStorageServer: started
[INFO]2026-03-28 02:44:25+0800: logging to ('/dev/stdout',)
[INFO]2026-03-28 02:44:25+0800: use event loop: builtin
[INFO]2026-03-28 02:44:25+0800: ConfigController: scheduler_address = tcp://127.0.0.1:34295
[INFO]2026-03-28 02:44:25+0800: ConfigController: object_storage_address = tcp://127.0.0.1:51993
[INFO]2026-03-28 02:44:25+0800: ConfigController: monitor_address = None
[INFO]2026-03-28 02:44:25+0800: ConfigController: protected = True
[INFO]2026-03-28 02:44:25+0800: ConfigController: max_number_of_tasks_waiting = -1
[INFO]2026-03-28 02:44:25+0800: ConfigController: client_timeout_seconds = 60
[INFO]2026-03-28 02:44:25+0800: ConfigController: worker_timeout_seconds = 60
[INFO]2026-03-28 02:44:25+0800: 